# 05 Circuit Pair Visualizations

Load finalized circuit artifacts from `configs.yaml` and write standalone HTML comparisons. Q/K/V/O components are bound into one attention-head node for visualization.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from circuit_discovery.run import load_configs
from circuit_discovery.visualization import load_circuit, visualize_circuit_comparison

configs = load_configs()
artifacts = configs["artifacts"]
out_dir = PROJECT_ROOT / configs["paths"]["visualization_output_dir"]
out_dir.mkdir(parents=True, exist_ok=True)


def circuit_from_config(*keys):
    value = artifacts
    for key in keys:
        value = value[key]
    return load_circuit(PROJECT_ROOT / value)


def save(fig, filename):
    path = out_dir / filename
    fig.write_html(path)
    print(path.relative_to(PROJECT_ROOT))


In [ ]:
fig = visualize_circuit_comparison(
    circuit_from_config("discogp", "circuits", "seed_42"),
    circuit_from_config("discogp", "circuits", "seed_42_overlap_ref_seed_42"),
    title="OASR Overlap Penalty",
    label_a="seed 42 edges",
    label_b="overlap-penalized edges",
)
save(fig, "01_oasr_alternative_sheaves.html")
fig


In [ ]:
fig = visualize_circuit_comparison(
    circuit_from_config("acdc", "circuits", "fixed_order"),
    circuit_from_config("acdc", "circuits", "random_per_layer_order_seed_42"),
    title="ACDC Traversal Ordering",
    label_a="fixed-order edges",
    label_b="random-order edges",
)
save(fig, "02_acdc_order_comparison.html")
fig


In [ ]:
k = artifacts["eap"]["selected_top_k"][-1]
path_template = artifacts["eap"]["path_template"]
fig = visualize_circuit_comparison(
    load_circuit(PROJECT_ROOT / path_template.format(condition="normal_order_42", top_k=k)),
    load_circuit(PROJECT_ROOT / path_template.format(condition="resampled_order_43", top_k=k)),
    title=f"EAP Name Resampling, top {k}",
    label_a="normal-order edges",
    label_b="resampled-name edges",
)
save(fig, "03_eap_name_sensitivity.html")
fig

In [ ]:
fig = visualize_circuit_comparison(
    circuit_from_config("edge_pruning", "circuits", "kl", "seed_42"),
    circuit_from_config("edge_pruning", "circuits", "kl", "seed_43"),
    title="Edge Pruning with KL Objective",
    label_a="seed 42 edges",
    label_b="seed 43 edges",
)
save(fig, "04_edge_pruning_kl_seed_comparison.html")
fig


In [ ]:
fig = visualize_circuit_comparison(
    circuit_from_config("edge_pruning", "circuits", "two_label", "seed_42"),
    circuit_from_config("edge_pruning", "circuits", "two_label", "seed_43"),
    title="Edge Pruning with Two-label Objective",
    label_a="seed 42 edges",
    label_b="seed 43 edges",
)
save(fig, "05_edge_pruning_ce_seed_comparison.html")
fig
